# 02: Kaggle inference with the frozen bundle `v2-A7` (internet OFF, GPU when available)

**Inputs.** All inputs come from attached Kaggle inputs. There is no clone, no Drive and no network:

* the **runtime source** dataset: this GitHub runtime repo uploaded as a Dataset;
* the **frozen bundle** dataset: the complete `bundle/` directory;
* the **competition** data;
* optionally, a wheel dataset holding `lightgbm-4.7.0-*.whl`.

**Stages:**

1. Discover the inputs.
2. Detect the GPU.
3. Validate the frozen bundle: manifest, `CONFIG_HASH`, calibration contract and identity.
4. Put `bundle/code` first on `sys.path`.
5. Run the self-test.
6. Choose a validated backend.
7. Run inference.
8. Validate the submission.
9. Write `/kaggle/working/submission.csv` and `run_report.json`.

**GPU policy.** A GPU backend is used only if the frozen bundle implements one, declares it parity-validated and
passes the self-test. Bundle v2-A7 ships none, so it runs on the validated CPU backend, and the report says why.
Submitting is a manual Kaggle action.

In [ ]:
# 1. configuration + discover the runtime-source dataset (the bundle + competition files are discovered by schema in stage 3)
import json, os, subprocess, sys, time
from pathlib import Path

DEVICE_PREFERENCE = 'gpu'          # prefer a GPU backend IF validated
REQUIRE_GPU = False                # production: a validated CPU backend is fine
EXPECTED_CONFIG_HASH = None        # optional: the CONFIG_HASH of YOUR export, to pin it
INPUT_ROOT = Path(os.environ.get('CASMI_KAGGLE_INPUT', '/kaggle/input'))
WORK_DIR = Path(os.environ.get('CASMI_KAGGLE_WORKING', '/kaggle/working'))

src_hits = sorted({p.parent.parent for p in INPUT_ROOT.rglob('casmi_runtime/__init__.py') if p.parent.parent.name == 'src'})
if len(src_hits) != 1:
    raise RuntimeError(f'attach exactly one runtime-source dataset (containing src/casmi_runtime), found {src_hits}')
RUNTIME_SRC = src_hits[0]
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))
print('runtime source:', RUNTIME_SRC)

In [ ]:
# 2. environment + GPU detection; lightgbm 4.7.0 from an ATTACHED wheel only (offline install, no index, no network)
from importlib.metadata import PackageNotFoundError, version


def _v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


LGBM_REQUIRED = '4.7.0'
if _v('lightgbm') != LGBM_REQUIRED:
    wheels = sorted(INPUT_ROOT.rglob(f'lightgbm-{LGBM_REQUIRED}-*.whl'))
    if wheels:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheels[0])], capture_output=True, text=True)
        print(r.stdout[-1500:], r.stderr[-1500:])
        r.check_returncode()
    else:
        print(f"WARNING: image lightgbm {_v('lightgbm')} != {LGBM_REQUIRED} and no wheel attached -- the run continues ONLY if the "
              'bundle self-test (fold scores at 1e-9) passes')
LGBM_VERSION = _v('lightgbm')
from casmi_runtime.accelerator import detect_accelerator
ACCEL = detect_accelerator()
print(json.dumps(ACCEL, indent=1, default=str))
if ACCEL['cuda_available']:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# 3-9. validate frozen bundle -> bundle/code first -> self-test -> validated backend -> inference -> validation -> outputs
from casmi_runtime.config import RuntimeConfig
from casmi_runtime.inference import run_frozen_inference
CFG = RuntimeConfig(runtime='kaggle', input_root=INPUT_ROOT, work_dir=WORK_DIR, device_preference=DEVICE_PREFERENCE, require_gpu=REQUIRE_GPU,
                    expected_config_hash=EXPECTED_CONFIG_HASH, extra={'lightgbm_version': LGBM_VERSION, 'runtime_src': str(RUNTIME_SRC)})
REPORT = run_frozen_inference(CFG)          # raises (and writes run_report_failed.json, no submission) on ANY failed check

In [ ]:
# inspect: the frozen bundle code ran; identity + validations; backend actually used and why
assert Path(REPORT['casmi_infer_source']).resolve().is_relative_to(Path(REPORT['frozen_code_dir']).resolve())
assert Path(REPORT['frozen_code_dir']).resolve().is_relative_to(Path(REPORT['bundle_dir']).resolve())
assert REPORT['self_test_status'] == 'PASS' and REPORT['submission_validation_status'] == 'PASS'
assert (WORK_DIR / 'submission.csv').exists() and (WORK_DIR / 'run_report.json').exists()
keys = ('runtime_platform', 'python_version', 'gpu_available', 'gpu_name', 'requested_device', 'actual_backend', 'gpu_used', 'numba_available',
        'numpy_fallback_used', 'fallback_reason', 'bundle_version', 'CONFIG_HASH', 'model_id', 'aggregator', 'calibration_hash',
        'casmi_infer_source', 'self_test_status', 'n_spectra', 'n_molecules', 'candidate_pair_count', 'runtime_seconds', 'peak_ram',
        'peak_gpu_memory', 'fallback_counts', 'exception_count', 'submission_rows', 'submission_validation_status', 'submission_sha256')
print(json.dumps({k: REPORT.get(k) for k in keys}, indent=1, default=str))
print('Outputs:', WORK_DIR / 'submission.csv', WORK_DIR / 'run_report.json', '-- submitting is a MANUAL Kaggle action.')